# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# Árboles: no necesitan escalado; solo imputación por mediana (datos futuros incompletos)
tree_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), feature_columns),
])

# LogReg y KNN: imputación + estandarización (sensibles a la escala)
scaled_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns),
])

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Si el scaler se ajusta con todo `X_train` antes de la CV, la media y la desviación estándar incluyen los datos de cada fold de validación. Así, información de validación se filtra al entrenamiento (*data leakage*) y la estimación de CV queda optimista. Dentro del `Pipeline`, `cross_validate` y `GridSearchCV` reajustan el scaler **solo con los folds de entrenamiento** de cada partición y lo aplican al fold de validación, igual que ocurrirá con datos nuevos en producción. Además, el artefacto final guarda scaler y modelo juntos.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline([
        ("preprocessing", clone(tree_preprocessor)),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("preprocessing", clone(scaled_preprocessor)),
        ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("preprocessing", clone(scaled_preprocessor)),
        ("classifier", KNeighborsClassifier()),
    ]),
}
list(models)

['decision_tree', 'logistic_regression', 'knn']

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipeline in models.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    rows.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "val_accuracy_mean": scores["test_score"].mean(),
        "val_accuracy_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(rows).sort_values("val_accuracy_mean", ascending=False).reset_index(drop=True)
initial_comparison.round(4)

,model,train_accuracy_mean,val_accuracy_mean,val_accuracy_std
0,logistic_regression,1.0000,0.9791,0.0277
1,knn,0.9771,0.9507,0.0353
2,decision_tree,1.0000,0.9022,0.0586


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {
        "classifier__criterion": ["gini", "entropy"],
        "classifier__max_depth": [2, 3, 4, 5, None],
        "classifier__min_samples_split": [2, 5, 10],
    },
    "logistic_regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100],
    },
    "knn": {
        "classifier__n_neighbors": [1, 3, 5, 7, 9, 11, 15],
        "classifier__weights": ["uniform", "distance"],
    },
}

searches = {}
rows = []
for name, pipeline in models.items():
    search = GridSearchCV(
        pipeline,
        param_grids[name],
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    best = search.best_index_
    rows.append({
        "model": name,
        "best_cv_accuracy": search.best_score_,
        "cv_std": search.cv_results_["std_test_score"][best],
        "train_accuracy": search.cv_results_["mean_train_score"][best],
        "best_params": search.best_params_,
    })

tuned_comparison = pd.DataFrame(rows).sort_values(["best_cv_accuracy", "cv_std"], ascending=[False, True]).reset_index(drop=True)
tuned_comparison

,model,best_cv_accuracy,cv_std,train_accuracy,best_params
0,logistic_regression,0.979064,0.027713,1.000000,{'classifier__C': 1}
1,knn,0.957882,0.040520,0.964773,"{'classifier__n_neighbors': 7, 'classifier__we..."
2,decision_tree,0.902217,0.058617,0.998246,"{'classifier__criterion': 'gini', 'classifier_..."


## 6. Selección y persistencia del campeón

In [8]:
# Mayor best_score_ de CV; en caso de empate, menor desviación estándar (orden de tuned_comparison)
champion_name = tuned_comparison.loc[0, "model"]
champion = searches[champion_name].best_estimator_
print("Campeón:", champion_name, "| CV accuracy:", round(searches[champion_name].best_score_, 4))

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "selection_metric": "accuracy",
    "cv_folds": 5,
    "cv_score": float(searches[champion_name].best_score_),
    "cv_std": float(tuned_comparison.loc[0, "cv_std"]),
    "best_params": searches[champion_name].best_params_,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, default=str), encoding="utf-8"
)

cv_results = pd.concat(
    [pd.DataFrame(s.cv_results_).assign(model=name) for name, s in searches.items()],
    ignore_index=True,
)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
training_metadata

Campeón: logistic_regression | CV accuracy: 0.9791


{'champion': 'logistic_regression',
 'selection_metric': 'accuracy',
 'cv_folds': 5,
 'cv_score': 0.9790640394088669,
 'cv_std': 0.027713464392187602,
 'best_params': {'classifier__C': 1}}

In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Se elige **regresión logística (`C=1`)**:
>
> - **Desempeño:** mejor accuracy de CV, 0.979, frente a 0.958 de KNN (`k=7`, `uniform`) y 0.902 del árbol (`gini`, `max_depth=4`).
> - **Variabilidad:** menor desviación entre folds (0.028 frente a 0.041 y 0.059).
> - **Sobreajuste:** accuracy de entrenamiento 1.00 frente a 0.979 en validación, una brecha pequeña de unos 2 puntos. El árbol tiene la brecha más grande (≈1.00 frente a 0.90).
> - **Complejidad:** modelo lineal con 3×13 coeficientes más interceptos, rápido de entrenar y predecir.
> - **Interpretabilidad:** los coeficientes (sobre variables estandarizadas) indican qué variables empujan hacia cada clase, con la advertencia de la multicolinealidad del Challenge 1.
>
> La selección se hizo solo con validación cruzada sobre `train.csv`; `test.csv` no se cargó en este notebook.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.